<a href="https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/notebooks/z2h-06-backprop-ninja-wip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## makemore: becoming a backprop ninja

In [ ]:
# there no change change in the first several cells from last lecture

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [ ]:
# download the names.txt file from github
!wget https://raw.githubusercontent.com/karpathy/makemore/master/names.txt

--2024-09-24 12:28:18--  https://raw.githubusercontent.com/karpathy/makemore/master/names.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 228145 (223K) [text/plain]
Saving to: ‘names.txt’

names.txt           100%[===================>] 222.80K  --.-KB/s    in 0.04s   

2024-09-24 12:28:18 (6.20 MB/s) - ‘names.txt’ saved [228145/228145]



In [ ]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
print(len(words))
print(max(len(w) for w in words))
print(words[:8])

32033
15
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']


In [ ]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [ ]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?

def build_dataset(words):
  X, Y = [], []

  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,  Ytr  = build_dataset(words[:n1])     # 80%
Xdev, Ydev = build_dataset(words[n1:n2])   # 10%
Xte,  Yte  = build_dataset(words[n2:])     # 10%

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [ ]:
# ok biolerplate done, now we get to the action:

In [ ]:
# utility function we will use later when comparing manual gradients to PyTorch gradients
def cmp(s, dt, t):
  ex = torch.all(dt == t.grad).item()
  app = torch.allclose(dt, t.grad)
  maxdiff = (dt - t.grad).abs().max().item()
  print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [ ]:
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 64 # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647) # for reproducibility
C  = torch.randn((vocab_size, n_embd),            generator=g)
# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden,                        generator=g) * 0.1 # using b1 just for fun, it's useless because of BN
# Layer 2
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.1
b2 = torch.randn(vocab_size,                      generator=g) * 0.1
# BatchNorm parameters
bngain = torch.randn((1, n_hidden))*0.1 + 1.0
bnbias = torch.randn((1, n_hidden))*0.1

# Note: I am initializating many of these parameters in non-standard ways
# because sometimes initializating with e.g. all zeros could mask an incorrect
# implementation of the backward pass.

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True

4137


In [ ]:
batch_size = 32
n = batch_size # a shorter variable also, for convenience
# construct a minibatch
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y

In [ ]:
"""
# forward pass, "chunkated" into smaller steps that are possible to backward one at a time

#Xb = (32,3)
emb = C[Xb] # embed the characters into vectors (32, 3, 10)
embcat = emb.view(emb.shape[0], -1) # concatenate the vectors (32, 30)
# Linear layer 1
hprebn = embcat @ W1 + b1 # hidden layer pre-activation (32, 30) @ (30, 64) + (64) = (32, 64)
# BatchNorm layer
bnmeani = 1/n*hprebn.sum(0, keepdim=True) # (1, 64)
bndiff = hprebn - bnmeani # (32, 64) - (1, 64) = (32, 64)
bndiff2 = bndiff**2 # (32, 64)
bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n)
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias
# Non-linearity
h = torch.tanh(hpreact) # hidden layer
# Linear layer 2
logits = h @ W2 + b2 # output layer
# cross entropy loss (same as F.cross_entropy(logits, Yb))
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()

# PyTorch backward pass
for p in parameters:
  p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv, # afaik there is no cleaner way
          norm_logits, logit_maxes, logits, h, hpreact, bnraw,
         bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,
         embcat, emb]:
  t.retain_grad()
loss.backward()
loss
"""

'\n# forward pass, "chunkated" into smaller steps that are possible to backward one at a time\n\n#Xb = (32,3)\nemb = C[Xb] # embed the characters into vectors (32, 3, 10)\nembcat = emb.view(emb.shape[0], -1) # concatenate the vectors (32, 30)\n# Linear layer 1\nhprebn = embcat @ W1 + b1 # hidden layer pre-activation (32, 30) @ (30, 64) + (64) = (32, 64)\n# BatchNorm layer\nbnmeani = 1/n*hprebn.sum(0, keepdim=True) # (1, 64)\nbndiff = hprebn - bnmeani # (32, 64) - (1, 64) = (32, 64)\nbndiff2 = bndiff**2 # (32, 64)\nbnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel\'s correction (dividing by n-1, not n)\nbnvar_inv = (bnvar + 1e-5)**-0.5\nbnraw = bndiff * bnvar_inv\nhpreact = bngain * bnraw + bnbias\n# Non-linearity\nh = torch.tanh(hpreact) # hidden layer\n# Linear layer 2\nlogits = h @ W2 + b2 # output layer\n# cross entropy loss (same as F.cross_entropy(logits, Yb))\nlogit_maxes = logits.max(1, keepdim=True).values\nnorm_logits = logits - logit_maxes # subtract max for numer

In [ ]:
# Xb: Input batch of data (32 samples, 3 elements per sample, possibly representing tokens/characters)
# Let's assume Xb is a batch of input indices of shape (32, 3), where 32 is the batch size,
# and 3 is the number of tokens per sample.

# Embedding lookup: converting input indices (characters) into vectors.
# C is the embedding matrix that maps characters to vectors.
emb = C[Xb]  # The result is of shape (32, 3, 10), where each character is mapped to a 10-dimensional vector.

# Flattening the embedded vectors for further processing.
embcat = emb.view(emb.shape[0], -1)  # Flatten the 3 vectors of size 10 into a single vector (32, 30).

# Linear Layer 1: Fully connected layer
# W1: Weights matrix of shape (30, 64), b1: bias vector of shape (64).
# hprebn: Pre-activation of the hidden layer, output of the first linear transformation.
hprebn = embcat @ W1 + b1  # Matrix multiplication between input (32, 30) and weights (30, 64) + bias (64) = (32, 64)

# Batch Normalization (manual implementation):
# Step 1: Compute the mean for each feature across the batch (mean along the batch dimension).
bnmeani = 1/n * hprebn.sum(0, keepdim=True)  # Mean across the batch, resulting shape (1, 64).

# Step 2: Subtract the mean from each element to center the data.
bndiff = hprebn - bnmeani  # Subtract the mean, resulting shape (32, 64).

# Step 3: Square the differences to compute variance.
bndiff2 = bndiff ** 2  # Element-wise square of differences, shape (32, 64).

# Step 4: Compute the variance (Bessel's correction by dividing by n-1).
bnvar = 1/(n-1) * bndiff2.sum(0, keepdim=True)  # Variance, shape (1, 64).

# Step 5: Compute the inverse standard deviation (with small epsilon for numerical stability).
bnvar_inv = (bnvar + 1e-5) ** -0.5  # Inverse of the standard deviation, shape (1, 64).

# Step 6: Normalize the data by multiplying the centered data by the inverse std deviation.
bnraw = bndiff * bnvar_inv  # Normalized batch (32, 64).

# Step 7: Scale and shift the normalized data using learnable parameters (batch normalization gain and bias).
hpreact = bngain * bnraw + bnbias  # Apply gain and bias to the normalized data, shape (32, 64).

# Non-linearity: Apply a non-linear activation function (tanh in this case).
h = torch.tanh(hpreact)  # Non-linear activation, output shape (32, 64).

# Linear Layer 2: Fully connected layer
# W2: Weights matrix of shape (64, num_classes), b2: bias vector of shape (num_classes).
logits = h @ W2 + b2  # Compute logits, shape (32, num_classes), where each sample has a vector of class scores.

# Cross-entropy loss calculation (manual implementation):
# Step 1: Stability trick - subtract the max logit from each logit for numerical stability.
logit_maxes = logits.max(1, keepdim=True).values  # Max logit per sample (32, 1).
norm_logits = logits - logit_maxes  # Subtract max logit for numerical stability, shape (32, num_classes).

# Step 2: Compute exponentiated logits (exp(logits)).
counts = norm_logits.exp()  # Exponentiate normalized logits, shape (32, num_classes).

# Step 3: Compute the sum of exponentiated logits for each sample.
counts_sum = counts.sum(1, keepdims=True)  # Sum of exp(logits) for each sample, shape (32, 1).

# Step 4: Inverse of the sum (for probability normalization).
counts_sum_inv = counts_sum ** -1  # Inverse of the sum, shape (32, 1).

# Step 5: Compute softmax probabilities (normalized logits).
probs = counts * counts_sum_inv  # Normalize the logits to get probabilities, shape (32, num_classes).

# Step 6: Compute the log of probabilities.
logprobs = probs.log()  # Take log of the probabilities, shape (32, num_classes).

# Step 7: Compute the negative log-likelihood for the true class labels (cross-entropy loss).
# Yb: True class labels for the batch of size 32.
loss = -logprobs[range(n), Yb].mean()  # Select log-probabilities corresponding to the correct class, then take the mean.

# Backpropagation (PyTorch backward pass):

# Step 1: Zero the gradients for all parameters (important to avoid accumulation from previous backward passes).
for p in parameters:
    p.grad = None  # Set gradients to None for all parameters.

# Step 2: Retain gradients for all intermediate tensors that we care about.
# We retain gradients for every intermediate step we want to inspect, which is useful for debugging or analyzing.
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv,  # All tensors up to the final loss computation.
          norm_logits, logit_maxes, logits, h, hpreact, bnraw,  # Backward pass needs these tensors for gradient calculation.
          bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,  # Batch normalization tensors that require gradients.
          embcat, emb]:  # Embedded inputs also require gradients.
    t.retain_grad()  # Retain gradients for these tensors.

# Step 3: Perform the actual backward pass to compute gradients of all parameters with respect to the loss.
loss.backward()  # Backpropagate the loss to compute all gradients.

In [ ]:
logprobs[range(n), Yb].shape

torch.Size([32])

In [ ]:
# emb = C[Xb]
# embcat = emb.view(emb.shape[0], -1)
# hprebn = embcat @ W1 + b1
# bnmeani = 1/n * hprebn.sum(0, keepdim=True)
# bndiff = hprebn - bnmeani
# bndiff2 = bndiff ** 2
# bnvar = 1/(n-1) * bndiff2.sum(0, keepdim=True)
# bnvar_inv = (bnvar + 1e-5) ** -0.5
# bnraw = bndiff * bnvar_inv
# hpreact = bngain * bnraw + bnbias
# h = torch.tanh(hpreact)
# logits = h @ W2 + b2
# logit_maxes = logits.max(1, keepdim=True).values
# norm_logits = logits - logit_maxes
# counts = norm_logits.exp()

# counts_sum = counts.sum(1, keepdims=True)
# counts_sum_inv = counts_sum ** -1
# probs = counts * counts_sum_inv
# logprobs = probs.log()
# loss = -logprobs[range(n), Yb].mean()

# loss = -logprobs[range(n), Yb].mean()
dloss_dlogprobs = torch.zeros_like(logprobs)
dloss_dlogprobs[range(n), Yb] = -1 / logprobs.shape[0] # this is dloss/dlogprobs
cmp('logprobs', dloss_dlogprobs, logprobs)

# logprobs = probs.log()
dlogprobs_dprobs = (1 / probs) # local derivative
dloss_dprobs = dlogprobs_dprobs * dloss_dlogprobs # chain rule

# probs = counts * counts_sum_inv
dprobs_dcounts = counts_sum_inv
dloss_dcounts1 = dprobs_dcounts * dloss_dprobs

# probs = counts * counts_sum_inv
dprobs_dcountssuminv = counts
dloss_dcountssuminv = (dprobs_dcountssuminv * dloss_dprobs).sum(1, keepdim=True)
cmp('counts_sum_inv', dloss_dcountssuminv, counts_sum_inv)

# counts_sum_inv = counts_sum ** -1
dcountssuminv_dcountsum = -counts_sum**-2
dloss_dcountsum = dcountssuminv_dcountsum * dloss_dcountssuminv
cmp('counts_sum', dloss_dcountsum, counts_sum)

# probs = counts * counts_sum_inv
dloss_dcounts2 = torch.ones_like(counts) * dloss_dcountsum
dloss_dcounts = dloss_dcounts1 + dloss_dcounts2
cmp('counts', dloss_dcounts, counts)

#
# cmp('norm_logits', dnorm_logits, norm_logits)
# cmp('logit_maxes', dlogit_maxes, logit_maxes)
# cmp('logits', dlogits, logits)
# cmp('h', dh, h)
# cmp('W2', dW2, W2)
# cmp('b2', db2, b2)
# cmp('hpreact', dhpreact, hpreact)
# cmp('bngain', dbngain, bngain)
# cmp('bnbias', dbnbias, bnbias)
# cmp('bnraw', dbnraw, bnraw)
# cmp('bnvar_inv', dbnvar_inv, bnvar_inv)
# cmp('bnvar', dbnvar, bnvar)
# cmp('bndiff2', dbndiff2, bndiff2)
# cmp('bndiff', dbndiff, bndiff)
# cmp('bnmeani', dbnmeani, bnmeani)
# cmp('hprebn', dhprebn, hprebn)
# cmp('embcat', dembcat, embcat)
# cmp('W1', dW1, W1)
# cmp('b1', db1, b1)
# cmp('emb', demb, emb)
# cmp('C', dC, C)



#dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)
#dcounts = counts_sum_inv * dprobs
#dcounts_sum = (-counts_sum**-2) * dcounts_sum_inv

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0
counts          | exact: True  | approximate: True  | maxdiff: 0.0


In [ ]:
hprebn.sum(0, keepdims=True) / 32

tensor([[-0.2923, -0.0526, -0.6841,  0.7001, -0.5036,  0.4972, -0.0668, -0.1110,
         -0.4643,  0.0302,  0.6675, -0.3830, -0.4715, -0.3956, -0.5209,  0.3950,
          0.0292, -1.6880,  0.3933,  0.8486, -0.6091, -1.2410, -0.0519, -0.2812,
          0.2267,  1.3896, -0.6374,  0.3277, -0.5348,  1.1896,  0.3176,  0.6083,
          0.7051, -0.5859, -0.2753,  1.9107, -1.1769, -0.7579,  0.1236,  0.4828,
          0.2302,  0.2900,  0.5919, -1.0644, -0.2177,  0.7055,  0.4917, -0.3559,
          0.6701,  1.5317, -0.5005, -0.2265,  1.7797,  0.6933,  1.5666, -1.1398,
         -0.4648, -0.8345,  0.6744, -0.1985, -1.3676, -0.5960,  0.1881,  0.7816]],
       grad_fn=<DivBackward0>)

In [ ]:
Yb.shape

torch.Size([32])

In [ ]:
logprobs[range(n), Yb]

tensor([-3.9032, -3.2115, -3.6487, -3.1794, -4.1403, -3.4645, -3.1844, -4.0702,
        -3.1768, -4.4075, -3.0587, -1.6326, -2.8004, -3.0148, -2.9782, -3.1047,
        -3.8394, -2.9308, -3.6134, -3.3109, -2.7729, -2.9366, -4.5147, -4.1580,
        -3.4183, -3.0215, -2.9829, -3.8168, -2.7294, -3.5097, -3.2025, -3.3617],
       grad_fn=<IndexBackward0>)

What is the derivative of logprobs:

Since `loss = -logprobs[range(n), Yb].mean()` and `-logprobs[range(n), Yb]` is a just a list of values, then it means that `loss = - (a + b + c + d ...) / count` which means that if were three items long then: `loss = (-1/3a) + (-1/3b) + (-1/3c)`. If you calculate the dloss/da = -1/3, this is because every other constant is zero when dloss/da, and if you apply the rule then `a` disappears because the `a` exponent is 1. Intuitively


This means that the loss in respect to any parameter is -1/count. So `dlogprobs` is the same shape as logprobs filled with zeros for the unplucked parameters (because these were not used for the loss, therefore don't impact it, therefore have zero gradient), and -1/count for all plucked values.

In [ ]:
dlogprobs = torch.zeros_like(logprobs) #
dlogprobs[range(n), Yb] = -1 / logprobs.shape[0]
cmp('logprobs', dlogprobs, logprobs)

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0


Next step is `dprobs`. For that we want to know how much probs contributed to changes in `logprobs`. `logprobs = probs.log()` so we need to know the derivative of log. So `logprobs = 1/probs`


The derivative of the natural logarithm function, \(\log(x)\), typically written as \(\ln(x)\), is:

\[
\frac{d}{dx} \ln(x) = \frac{1}{x}, \quad \text{for} \, x > 0.
\]

If you are referring to the logarithm with a different base, say \(\log_b(x)\), where \(b\) is the base of the logarithm, the derivative is:

\[
\frac{d}{dx} \log_b(x) = \frac{1}{x \ln(b)}, \quad \text{for} \, x > 0.
\]

In both cases, the derivative is defined for \(x > 0\) since the logarithm is only defined for positive values of \(x\).

Then we need to multiply it with `dlogprobs` because do know `dloss/dprobs = dlogprobs/dloss * dloss/dlogprobs`.

In [ ]:
dprobs = 1 / probs * dlogprobs
cmp('probs', dprobs, probs)

probs           | exact: True  | approximate: True  | maxdiff: 0.0


Next in line is `probs = counts * counts_sum_inv`

```
# Cross-entropy loss calculation (manual implementation):
# Step 1: Stability trick - subtract the max logit from each logit for numerical stability.
logit_maxes = logits.max(1, keepdim=True).values  # Max logit per sample (32, 1).
norm_logits = logits - logit_maxes  # Subtract max logit for numerical stability, shape (32, num_classes).

# Step 2: Compute exponentiated logits (exp(logits)).
counts = norm_logits.exp()  # Exponentiate normalized logits, shape (32, num_classes).

# Step 3: Compute the sum of exponentiated logits for each sample.
counts_sum = counts.sum(1, keepdims=True)  # Sum of exp(logits) for each sample, shape (32, 1).

# Step 4: Inverse of the sum (for probability normalization).
counts_sum_inv = counts_sum ** -1  # Inverse of the sum, shape (32, 1).

# Step 5: Compute softmax probabilities (normalized logits).
probs = counts * counts_sum_inv  # Normalize the logits to get probabilities, shape (32, num_classes).


# c = a * b, but with tensors:
# a[3x3] * b[3,1] -->
# a11*b1 a12*b1 a13*b1
# a21*b2 a22*b2 a23*b2
# a31*b3 a32*b3 a33*b3
# c[3x3]
```



In [ ]:
counts.shape, counts_sum_inv.shape

(torch.Size([32, 27]), torch.Size([32, 1]))

In [ ]:
dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)

counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0


In [ ]:
# counts_sum = counts.sum(1, keepdims=True)  # Sum of exp(logits) for each sample, shape (32, 1).
dcounts_sum = None
cmp('counts_sum', dcounts_sum, counts_sum)

TypeError: all() received an invalid combination of arguments - got (bool), but expected one of:
 * (Tensor input, *, Tensor out = None)
 * (Tensor input, tuple of ints dim = None, bool keepdim = False, *, Tensor out = None)
 * (Tensor input, int dim, bool keepdim = False, *, Tensor out = None)
 * (Tensor input, name dim, bool keepdim = False, *, Tensor out = None)




dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)

In [ ]:
# Exercise 1: backprop through the whole thing manually,
# backpropagating through exactly all of the variables
# as they are defined in the forward pass above, one by one


# TODO: I AM HERE!!!
# counts_sum_inv = counts_sum ** -1

# -----------------
# YOUR CODE HERE :)
# -----------------
#
# cmp('counts', dcounts, counts)
# cmp('norm_logits', dnorm_logits, norm_logits)
# cmp('logit_maxes', dlogit_maxes, logit_maxes)
# cmp('logits', dlogits, logits)
# cmp('h', dh, h)
# cmp('W2', dW2, W2)
# cmp('b2', db2, b2)
# cmp('hpreact', dhpreact, hpreact)
# cmp('bngain', dbngain, bngain)
# cmp('bnbias', dbnbias, bnbias)
# cmp('bnraw', dbnraw, bnraw)
# cmp('bnvar_inv', dbnvar_inv, bnvar_inv)
# cmp('bnvar', dbnvar, bnvar)
# cmp('bndiff2', dbndiff2, bndiff2)
# cmp('bndiff', dbndiff, bndiff)
# cmp('bnmeani', dbnmeani, bnmeani)
# cmp('hprebn', dhprebn, hprebn)
# cmp('embcat', dembcat, embcat)
# cmp('W1', dW1, W1)
# cmp('b1', db1, b1)
# cmp('emb', demb, emb)
# cmp('C', dC, C)

In [ ]:
# Exercise 2: backprop through cross_entropy but all in one go
# to complete this challenge look at the mathematical expression of the loss,
# take the derivative, simplify the expression, and just write it out

# forward pass

# before:
# logit_maxes = logits.max(1, keepdim=True).values
# norm_logits = logits - logit_maxes # subtract max for numerical stability
# counts = norm_logits.exp()
# counts_sum = counts.sum(1, keepdims=True)
# counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
# probs = counts * counts_sum_inv
# logprobs = probs.log()
# loss = -logprobs[range(n), Yb].mean()

# now:
loss_fast = F.cross_entropy(logits, Yb)
print(loss_fast.item(), 'diff:', (loss_fast - loss).item())

In [ ]:
# backward pass

# -----------------
# YOUR CODE HERE :)
dlogits = None # TODO. my solution is 3 lines
# -----------------

#cmp('logits', dlogits, logits) # I can only get approximate to be true, my maxdiff is 6e-9

In [ ]:
# Exercise 3: backprop through batchnorm but all in one go
# to complete this challenge look at the mathematical expression of the output of batchnorm,
# take the derivative w.r.t. its input, simplify the expression, and just write it out
# BatchNorm paper: https://arxiv.org/abs/1502.03167

# forward pass

# before:
# bnmeani = 1/n*hprebn.sum(0, keepdim=True)
# bndiff = hprebn - bnmeani
# bndiff2 = bndiff**2
# bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n)
# bnvar_inv = (bnvar + 1e-5)**-0.5
# bnraw = bndiff * bnvar_inv
# hpreact = bngain * bnraw + bnbias

# now:
hpreact_fast = bngain * (hprebn - hprebn.mean(0, keepdim=True)) / torch.sqrt(hprebn.var(0, keepdim=True, unbiased=True) + 1e-5) + bnbias
print('max diff:', (hpreact_fast - hpreact).abs().max())

In [ ]:
# backward pass

# before we had:
# dbnraw = bngain * dhpreact
# dbndiff = bnvar_inv * dbnraw
# dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)
# dbnvar = (-0.5*(bnvar + 1e-5)**-1.5) * dbnvar_inv
# dbndiff2 = (1.0/(n-1))*torch.ones_like(bndiff2) * dbnvar
# dbndiff += (2*bndiff) * dbndiff2
# dhprebn = dbndiff.clone()
# dbnmeani = (-dbndiff).sum(0)
# dhprebn += 1.0/n * (torch.ones_like(hprebn) * dbnmeani)

# calculate dhprebn given dhpreact (i.e. backprop through the batchnorm)
# (you'll also need to use some of the variables from the forward pass up above)

# -----------------
# YOUR CODE HERE :)
dhprebn = None # TODO. my solution is 1 (long) line
# -----------------

cmp('hprebn', dhprebn, hprebn) # I can only get approximate to be true, my maxdiff is 9e-10

In [ ]:
# Exercise 4: putting it all together!
# Train the MLP neural net with your own backward pass

# init
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 200 # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647) # for reproducibility
C  = torch.randn((vocab_size, n_embd),            generator=g)
# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden,                        generator=g) * 0.1
# Layer 2
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.1
b2 = torch.randn(vocab_size,                      generator=g) * 0.1
# BatchNorm parameters
bngain = torch.randn((1, n_hidden))*0.1 + 1.0
bnbias = torch.randn((1, n_hidden))*0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True

# same optimization as last time
max_steps = 200000
batch_size = 32
n = batch_size # convenience
lossi = []

# use this context manager for efficiency once your backward pass is written (TODO)
#with torch.no_grad():

# kick off optimization
for i in range(max_steps):

  # minibatch construct
  ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
  Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y

  # forward pass
  emb = C[Xb] # embed the characters into vectors
  embcat = emb.view(emb.shape[0], -1) # concatenate the vectors
  # Linear layer
  hprebn = embcat @ W1 + b1 # hidden layer pre-activation
  # BatchNorm layer
  # -------------------------------------------------------------
  bnmean = hprebn.mean(0, keepdim=True)
  bnvar = hprebn.var(0, keepdim=True, unbiased=True)
  bnvar_inv = (bnvar + 1e-5)**-0.5
  bnraw = (hprebn - bnmean) * bnvar_inv
  hpreact = bngain * bnraw + bnbias
  # -------------------------------------------------------------
  # Non-linearity
  h = torch.tanh(hpreact) # hidden layer
  logits = h @ W2 + b2 # output layer
  loss = F.cross_entropy(logits, Yb) # loss function

  # backward pass
  for p in parameters:
    p.grad = None
  loss.backward() # use this for correctness comparisons, delete it later!

  # manual backprop! #swole_doge_meme
  # -----------------
  # YOUR CODE HERE :)
  dC, dW1, db1, dW2, db2, dbngain, dbnbias = None, None, None, None, None, None, None
  grads = [dC, dW1, db1, dW2, db2, dbngain, dbnbias]
  # -----------------

  # update
  lr = 0.1 if i < 100000 else 0.01 # step learning rate decay
  for p, grad in zip(parameters, grads):
    p.data += -lr * p.grad # old way of cheems doge (using PyTorch grad from .backward())
    #p.data += -lr * grad # new way of swole doge TODO: enable

  # track stats
  if i % 10000 == 0: # print every once in a while
    print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
  lossi.append(loss.log10().item())

  if i >= 100: # TODO: delete early breaking when you're ready to train the full net
    break

In [ ]:
# useful for checking your gradients
# for p,g in zip(parameters, grads):
#   cmp(str(tuple(p.shape)), g, p)

In [ ]:
# calibrate the batch norm at the end of training

with torch.no_grad():
  # pass the training set through
  emb = C[Xtr]
  embcat = emb.view(emb.shape[0], -1)
  hpreact = embcat @ W1 + b1
  # measure the mean/std over the entire training set
  bnmean = hpreact.mean(0, keepdim=True)
  bnvar = hpreact.var(0, keepdim=True, unbiased=True)


In [ ]:
# evaluate train and val loss

@torch.no_grad() # this decorator disables gradient tracking
def split_loss(split):
  x,y = {
    'train': (Xtr, Ytr),
    'val': (Xdev, Ydev),
    'test': (Xte, Yte),
  }[split]
  emb = C[x] # (N, block_size, n_embd)
  embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
  hpreact = embcat @ W1 + b1
  hpreact = bngain * (hpreact - bnmean) * (bnvar + 1e-5)**-0.5 + bnbias
  h = torch.tanh(hpreact) # (N, n_hidden)
  logits = h @ W2 + b2 # (N, vocab_size)
  loss = F.cross_entropy(logits, y)
  print(split, loss.item())

split_loss('train')
split_loss('val')

In [ ]:
# I achieved:
# train 2.0718822479248047
# val 2.1162495613098145

In [ ]:
# sample from the model
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):

    out = []
    context = [0] * block_size # initialize with all ...
    while True:
      # forward pass
      emb = C[torch.tensor([context])] # (1,block_size,d)
      embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
      hpreact = embcat @ W1 + b1
      hpreact = bngain * (hpreact - bnmean) * (bnvar + 1e-5)**-0.5 + bnbias
      h = torch.tanh(hpreact) # (N, n_hidden)
      logits = h @ W2 + b2 # (N, vocab_size)
      # sample
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break

    print(''.join(itos[i] for i in out))